In [9]:
### Test code for group file

import pandas as pd

groups_file = r"D:\Expriment\Code\Python\fear_conditioning_V1.0\2PCIFA\groups\d1.csv"
groups = pd.read_csv(groups_file)
groups1 = groups.values.tolist()
for i, group in enumerate(groups1):
    for j, subject in enumerate(group):
        print(f"Group {i}, Subject {j}: {subject}")

Group 0, Subject 0: F:\Calcium-fear\raw data\20260324\Hab\2410\output\20260324_Hab_2410_ch1
Group 0, Subject 1: F:\Calcium-fear\raw data\20260324\EXT1\2410\output\20260324_EXT1_2410_ch1


In [ ]:
### Test code for loading and aligning running data with imaging data based on trigger signals and timestamps

from core.io import load_running_data
from matplotlib import pyplot as plt
import h5py
import numpy as np
import xml.etree.ElementTree as ET
import pandas as pd
from core.calculate_overall_dff import std_based_baseline_windows

data = r"F:\Calcium-fear\raw data\20260324\EXT1\2401\2401_EXT1_speed_20260325_172034_AST2_20260325_172034.ast2"
episode_file = r"F:\Calcium-fear\raw data\20260324\EXT1\2401\SyncData_0073\Episode_0000.h5"
real_time_xml = r"F:\Calcium-fear\raw data\20260324\EXT1\2401\SyncData_0073\ThorRealTimeDataSettings.xml"
experiment_xml_file = r"F:\Calcium-fear\raw data\20260324\EXT1\2401\2401_EXT1\Experiment.xml"
timestamp_path = r"F:\Calcium-fear\raw data\20260324\EXT1\2401\2401_EXT1_20260325_172034_timestamps.csv"
f_path = r"D:\Expriment\Code\Python\fear_conditioning_V1.0\2PCIFA\data\data_t2p\a2a\paired1\track2p\matched_suite2p\20260324_EXT1_2401_ch1\suite2p\plane0\F.npy"
fn_path = r"D:\Expriment\Code\Python\fear_conditioning_V1.0\2PCIFA\data\data_t2p\a2a\paired1\track2p\matched_suite2p\20260324_EXT1_2401_ch1\suite2p\plane0\Fneu.npy"

f = np.load(f_path)
fn = np.load(fn_path)
intensity = f - 0.7 * fn
[baseline_window, _, _, _] = std_based_baseline_windows(intensity, window_size=21)
baseline = intensity[baseline_window[0][0]:baseline_window[0][1]]
dff = (intensity - baseline.mean()) / baseline.mean()

speed_data, raw_time = load_running_data(data)
running_relative_time = (raw_time - raw_time[0])

with h5py.File(episode_file, 'r') as h5f:
    running_sync = np.array(h5f['/AI/Runningdata'])
    frame_out = np.array(h5f['/DI/FrameOut'])
    trigger_signal = np.array(h5f['/DI/Triggersignal'])

tree = ET.parse(real_time_xml)
root = tree.getroot()

daq = root.find(".//AcquireBoard[@active='1']")
if daq is None:
    raise RuntimeError("No active DAQ board found in ThorRealTimeDataSettings.xml")  

sr_node = daq.find(".//SampleRate[@enable='1']")
if sr_node is None:
    raise RuntimeError("No enabled <SampleRate> found in active DAQ board")

sample_rate = float(sr_node.get("rate"))

tree = ET.parse(experiment_xml_file)
root = tree.getroot()    
lsm = root.find('.//LSM[@name="ResonanceGalvo"]')

if lsm is not None:
    frame_rate = float(lsm.get('frameRate'))
    average_num = int(lsm.get('averageNum'))
else:
    print("LSM cannot find!", "WARNING")
    frame_rate = 10.0
    average_num = 1

streaming = root.find('.//Streaming[@enable="1"]')
if streaming is not None:
    frames = int(streaming.get('frames'))
else:
    print("Streaming cannot find!", "WARNING")
    frames = 1000
    
running_onset_idx = np.where(trigger_signal > 0.5)[0]
imaging_onset_idx = np.where(frame_out > 0.5)[0]

if len(running_onset_idx) > 0 and len(imaging_onset_idx) > 0:
    running_onset = running_onset_idx[0]
    diff = np.diff(imaging_onset_idx)
    breaks = np.where(diff > 100)
    imaging_session = []
    for i in range(len(breaks[0]) + 1):
        if i == 0:
            start = imaging_onset_idx[0]
        else:
            start = imaging_onset_idx[breaks[0][i-1] + 1]
        
        if i == len(breaks[0]) :
            end = imaging_onset_idx[-1]
        else:
            end = imaging_onset_idx[breaks[0][i]]
            
        imaging_session.append((start, end))
        
    for i, (start, end) in enumerate(imaging_session):
        duration = (end - start) / sample_rate
        threshold = 0.005
        imaging_time = frames / (frame_rate / average_num)
        if abs(duration - imaging_time) < threshold*imaging_time:
            imaging_onset = start
            relative_time1 = (running_onset - imaging_onset) / sample_rate
            break
else:
    relative_time1 = 0.0
    print("Trigger signals not found, using zero offset", "WARNING")

timestamps = pd.read_csv(timestamp_path)
exp_start = timestamps[(timestamps['Device'] == 'Experiment') &
                    (timestamps['Action'] == 'Start')]['Timestamp'].values
speed_start = timestamps[(timestamps['Device'] == 'Speed Sensor') &
                    (timestamps['Action'] == 'Start')]['Timestamp'].values
speed_end = timestamps[(timestamps['Device'] == 'Speed Sensor') &
                    (timestamps['Action'] == 'End')]['Timestamp'].values

if len(exp_start) > 0 and len(speed_start) > 0:
    relative_time2 = exp_start[0] - speed_start[0]
    duration = speed_end[0] - speed_start[0]

running_time = running_relative_time- relative_time2
imaging_time = np.arange(frames) / frame_rate * average_num - relative_time1 - relative_time2

valid_mask_running = (running_time >= 0) & (running_time <= duration)
valid_mask_imaging = (imaging_time >= 0) & (imaging_time <= duration)
aligned_running_data = speed_data[valid_mask_running]
aligned_dff = dff[:, valid_mask_imaging]

plt.figure()
plt.plot(frame_out, color="#3CFF00", label='frame out signal')
plt.plot(trigger_signal, color="#0004FF", label='trigger signal')
plt.axvline(running_onset, color='#000000', label='running onset')
plt.axvline(imaging_onset, color="#FF0000", label='imaging onset')
plt.legend(loc='upper left')
plt.show()

plt.figure(figsize=(16, 1))
plt.plot(aligned_running_data, color='gray', alpha=0.5)
plt.xlabel('Time (frames)')
plt.xlim(0, len(aligned_running_data))
plt.ylabel('Speed (cm/s)')
plt.show()

plt.figure(figsize=(16, 9))
shift = 5
for n in range(aligned_dff.shape[0]):
    i = aligned_dff[n, :]
    plt.plot(i + shift * n)

plt.xlabel('Time (frames)')
plt.xlim(0, aligned_dff.shape[1])
plt.ylabel('DF/F')
plt.show()